## 4. Reconstruction Evaluation Example

The official reconstruction evaluation should be performed with the TA evaluation script. The uploaded `utils_eval.py` contains the full implementation of:

- pixel correlation,
- SSIM,
- AlexNet-based perceptual identification,
- Inception-based identification,
- CLIP-based identification,
- EfficientNet correlation,
- SwAV correlation,
- and the final `eval_images(...)` wrapper.

Below, we inline that code into the notebook so that students can call the evaluation directly.

### Important note

This evaluation code uses pretrained vision models. On first execution, PyTorch or related packages may need to download pretrained weights. The code below is included as the **official evaluation example**, but it can be computationally expensive.

In [1]:
# Install required dependencies for evaluation
!pip install -q scikit-image
!pip install -q git+https://github.com/openai/CLIP.git

In [2]:
import numpy as np
import torch
from torchvision import transforms
from tqdm import tqdm
from torchvision.models.feature_extraction import create_feature_extractor
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity
from torchvision.models import inception_v3, Inception_V3_Weights
import clip
from torchvision.models import efficientnet_b1, EfficientNet_B1_Weights
import scipy as sp
import os
from PIL import Image
import datetime
import json
import glob


@torch.no_grad()
def two_way_identification(
    all_brain_recons,
    all_images,
    model,
    preprocess,
    feature_layer=None,
    return_avg=True,
    device: torch.device = torch.device("cpu"),
):
    preds = model(
        torch.stack([preprocess(recon) for recon in all_brain_recons], dim=0).to(device)
    )
    reals = model(
        torch.stack([preprocess(indiv) for indiv in all_images], dim=0).to(device)
    )
    if feature_layer is None:
        preds = preds.float().flatten(1).cpu().numpy()
        reals = reals.float().flatten(1).cpu().numpy()
    else:
        preds = preds[feature_layer].float().flatten(1).cpu().numpy()
        reals = reals[feature_layer].float().flatten(1).cpu().numpy()

    r = np.corrcoef(reals, preds)
    r = r[: len(all_images), len(all_images) :]
    congruents = np.diag(r)

    success = r < congruents
    success_cnt = np.sum(success, 0)

    if return_avg:
        perf = np.mean(success_cnt) / (len(all_images) - 1)
        return perf
    else:
        return success_cnt, len(all_images) - 1


def pixcorr(all_images, all_brain_recons):
    preprocess = transforms.Compose(
        [
            transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
        ]
    )

    all_images_flattened = preprocess(all_images).reshape(len(all_images), -1).cpu()
    all_brain_recons_flattened = (
        preprocess(all_brain_recons).reshape(len(all_brain_recons), -1).cpu()
    )

    corrsum = 0
    n = min(len(all_images_flattened), len(all_brain_recons_flattened))
    for i in tqdm(range(n)):
        corrsum += np.corrcoef(all_images_flattened[i], all_brain_recons_flattened[i])[
            0
        ][1]
    corrmean = corrsum / n

    pixcorr = corrmean
    return pixcorr


def ssim(all_images, all_brain_recons):
    preprocess = transforms.Compose(
        [
            transforms.Resize(425, interpolation=transforms.InterpolationMode.BILINEAR),
        ]
    )

    img_gray = rgb2gray(preprocess(all_images).permute((0, 2, 3, 1)).cpu())
    recon_gray = rgb2gray(preprocess(all_brain_recons).permute((0, 2, 3, 1)).cpu())

    ssim_score = []
    for im, rec in tqdm(zip(img_gray, recon_gray), total=len(all_images)):
        ssim_score.append(
            structural_similarity(
                rec,
                im,
                multichannel=True,
                gaussian_weights=True,
                sigma=1.5,
                use_sample_covariance=False,
                data_range=1.0,
            )
        )

    ssim = np.mean(ssim_score)
    return ssim


def alexnet(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    from torchvision.models import alexnet, AlexNet_Weights

    alex_weights = AlexNet_Weights.IMAGENET1K_V1

    alex_model = create_feature_extractor(
        alexnet(weights=alex_weights), return_nodes=["features.4", "features.11"]
    ).to(device)
    alex_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons.float(),
        all_images,
        alex_model,
        preprocess,
        "features.4",
        device=device,
    )
    alexnet2 = np.mean(all_per_correct)

    all_per_correct = two_way_identification(
        all_brain_recons.float(),
        all_images,
        alex_model,
        preprocess,
        "features.11",
        device=device,
    )
    alexnet5 = np.mean(all_per_correct)
    return alexnet2, alexnet5


def inception(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    weights = Inception_V3_Weights.DEFAULT
    inception_model = create_feature_extractor(
        inception_v3(weights=weights), return_nodes=["avgpool"]
    ).to(device)
    inception_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(342, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons,
        all_images,
        inception_model,
        preprocess,
        "avgpool",
        device=device,
    )

    inception = np.mean(all_per_correct)
    return inception


def clip_(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    clip_model, preprocess = clip.load("ViT-L/14", device=device)

    preprocess = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(
                mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711],
            ),
        ]
    )

    all_per_correct = two_way_identification(
        all_brain_recons,
        all_images,
        clip_model.encode_image,
        preprocess,
        None,
        device=device,
    )
    clip_ = np.mean(all_per_correct)
    return clip_


def effnet(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    weights = EfficientNet_B1_Weights.DEFAULT
    eff_model = create_feature_extractor(
        efficientnet_b1(weights=weights), return_nodes=["avgpool"]
    ).to(device)
    eff_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(255, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    gt = eff_model(preprocess(all_images))["avgpool"]
    gt = gt.reshape(len(gt), -1).cpu().numpy()
    fake = eff_model(preprocess(all_brain_recons))["avgpool"]
    fake = fake.reshape(len(fake), -1).cpu().numpy()

    effnet = np.array(
        [sp.spatial.distance.correlation(gt[i], fake[i]) for i in range(len(gt))]
    ).mean()
    return effnet


def swav(all_images, all_brain_recons, device: torch.device = torch.device("cpu")):
    swav_model = torch.hub.load("facebookresearch/swav:main", "resnet50")
    swav_model = create_feature_extractor(swav_model, return_nodes=["avgpool"]).to(
        device
    )
    swav_model.eval().requires_grad_(False)

    preprocess = transforms.Compose(
        [
            transforms.Resize(224, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    gt = swav_model(preprocess(all_images))["avgpool"]
    gt = gt.reshape(len(gt), -1).cpu().numpy()
    fake = swav_model(preprocess(all_brain_recons))["avgpool"]
    fake = fake.reshape(len(fake), -1).cpu().numpy()

    swav = np.array(
        [sp.spatial.distance.correlation(gt[i], fake[i]) for i in range(len(gt))]
    ).mean()
    return swav


def eval_images(
    real_images: torch.Tensor,
    fake_images: torch.Tensor,
    device: torch.device = torch.device("cpu"),
):
    real_images = real_images.to(device).float()
    fake_images = fake_images.to(device).float()

    pixcorrs = pixcorr(real_images, fake_images)
    ssims = ssim(real_images, fake_images)
    alex2, alex5 = alexnet(real_images, fake_images, device=device)
    inceptions = inception(real_images, fake_images, device=device)
    clips = clip_(real_images, fake_images, device=device)
    effnets = effnet(real_images, fake_images, device=device)
    swavs = swav(real_images, fake_images, device=device)

    return {
        "eval_pixcorr": pixcorrs.item(),
        "eval_ssim": ssims.item(),
        "eval_alex2": alex2.item(),
        "eval_alex5": alex5.item(),
        "eval_inception": inceptions.item(),
        "eval_clip": clips.item(),
        "eval_effnet": effnets.item(),
        "eval_swav": swavs.item(),
    }

### 4.1 Task 2 reconstruction evaluation for our submission

The cells below are only glue code for our submitted Task 2 pipeline. They do **not** change the official `eval_images(...)` logic above.

By default, the notebook looks for reconstructed images produced by our program under `task2/runs/.../generated_image/all/`, pairs them with the course test images by filename, and then calls the official evaluator directly.

If the reconstructed images have not been generated yet, set `RUN_TASK2_PIPELINE = True` in the next cell. This will run our Task 2 program first and then evaluate the generated images.

In [4]:
import subprocess
from pathlib import Path
from PIL import Image

# Set this to True only when the Task 2 images have not been generated yet.
# The full pipeline is expensive because it may train, align, generate, and evaluate.
RUN_TASK2_PIPELINE = False
TASK2_PIPELINE_STAGE = "all"  # one of: all, prepare, train, align, generate, eval, summary

TASK2_DIR = Path("task2")
TASK2_RUN_TAG = "full_tune"
TASK2_EXP_NAME = "intra-subject_cogcappro_EEGProjectLayer_multimodal_cogcap_list_ViT-H-14"
TASK2_SUBJECT = "sub-01"
TASK2_SEEDS = list(range(10))
TASK2_GENERATED_MODE = "all"
TASK2_IMAGE_SIZE = 256
TASK2_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COURSE_DATA_ROOT = Path("image-eeg-data")
REAL_IMAGE_ROOT = (
    COURSE_DATA_ROOT
    / "converted_for_cogcappro"
    / "ThingsEEG"
    / "Image_set_Resize"
    / "test_images"
)
if not REAL_IMAGE_ROOT.exists():
    REAL_IMAGE_ROOT = COURSE_DATA_ROOT / "test_images"

_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}


def run_task2_program(seed: int) -> None:
    """Run our Task 2 pipeline for one seed, using the submitted shell entrypoint."""
    env = os.environ.copy()
    env.update(
        {
            "RUN_TAG": TASK2_RUN_TAG,
            "SEED": str(seed),
            "SUBJECT": TASK2_SUBJECT,
            "MODALITY_MODE": TASK2_GENERATED_MODE,
        }
    )
    subprocess.run(
        ["bash", "scripts/run_full_experiment.sh", TASK2_PIPELINE_STAGE],
        cwd=TASK2_DIR,
        env=env,
        check=True,
    )


def _contains_images(path: Path) -> bool:
    return path.exists() and any(
        p.is_file() and p.suffix.lower() in _IMAGE_SUFFIXES for p in path.rglob("*")
    )


def resolve_task2_fake_root(seed: int) -> Path:
    """Find generated reconstruction images for a seed."""
    preferred = (
        TASK2_DIR
        / "runs"
        / TASK2_RUN_TAG
        / TASK2_EXP_NAME
        / f"{TASK2_SUBJECT}_seed{seed}"
        / "generated_image"
        / TASK2_GENERATED_MODE
    )
    candidates = [preferred]
    candidates.extend(
        sorted(
            TASK2_DIR.glob(
                f"runs/**/{TASK2_SUBJECT}_seed{seed}/generated_image/{TASK2_GENERATED_MODE}"
            ),
            key=lambda p: p.stat().st_mtime if p.exists() else 0,
            reverse=True,
        )
    )

    for candidate in candidates:
        if _contains_images(candidate):
            return candidate

    raise FileNotFoundError(
        f"No generated Task 2 images found for seed {seed}. "
        f"Expected images under paths like: {preferred}"
    )


def load_image_tensor(path: Path, image_size: int) -> torch.Tensor:
    with Image.open(path) as image:
        return transforms.Compose(
            [
                transforms.Resize(
                    (image_size, image_size),
                    interpolation=transforms.InterpolationMode.BILINEAR,
                ),
                transforms.ToTensor(),
            ]
        )(image.convert("RGB"))


def load_paired_reconstruction_tensors(
    real_root: Path,
    fake_root: Path,
    image_size: int,
) -> tuple[torch.Tensor, torch.Tensor, list[str]]:
    """Load real and reconstructed images paired by filename."""
    if not real_root.exists():
        raise FileNotFoundError(f"Real image root does not exist: {real_root}")
    if not fake_root.exists():
        raise FileNotFoundError(f"Generated image root does not exist: {fake_root}")

    real_by_name = {
        path.name: path
        for path in real_root.rglob("*")
        if path.is_file() and path.suffix.lower() in _IMAGE_SUFFIXES
    }
    fake_paths = [
        path
        for path in sorted(fake_root.rglob("*"))
        if path.is_file()
        and path.suffix.lower() in _IMAGE_SUFFIXES
        and path.name in real_by_name
    ]
    if not fake_paths:
        raise FileNotFoundError(
            f"No filename-matched image pairs found between {real_root} and {fake_root}"
        )

    names = [path.name for path in fake_paths]
    real_images = torch.stack(
        [load_image_tensor(real_by_name[name], image_size) for name in names]
    )
    fake_images = torch.stack(
        [load_image_tensor(path, image_size) for path in fake_paths]
    )
    return real_images, fake_images, names


def evaluate_task2_seed(seed: int) -> dict:
    fake_root = resolve_task2_fake_root(seed)
    real_images, fake_images, matched_names = load_paired_reconstruction_tensors(
        real_root=REAL_IMAGE_ROOT,
        fake_root=fake_root,
        image_size=TASK2_IMAGE_SIZE,
    )
    metrics = eval_images(
        real_images=real_images,
        fake_images=fake_images,
        device=torch.device(TASK2_DEVICE),
    )
    metrics["seed"] = seed
    metrics["matched_images"] = len(matched_names)
    metrics["fake_root"] = str(fake_root)
    return metrics

In [ ]:
# Official Task 2 evaluation over generated reconstruction images.
# This calls the official eval_images(...) function defined above.
if RUN_TASK2_PIPELINE:
    for seed in TASK2_SEEDS:
        print(f"Running Task 2 pipeline for seed={seed} ...")
        run_task2_program(seed)

reconstruction_results = []
missing_seeds = []
for seed in TASK2_SEEDS:
    try:
        metrics = evaluate_task2_seed(seed)
        reconstruction_results.append(metrics)
    except FileNotFoundError as exc:
        missing_seeds.append((seed, str(exc)))
        print(f"[skip] seed={seed:02d}: {exc}")

if not reconstruction_results:
    raise FileNotFoundError(
        "No Task 2 reconstruction outputs were found. "
        "Run the Task 2 program first, or set RUN_TASK2_PIPELINE = True above."
    )

reconstruction_metric_dicts = [
    {key: value for key, value in result.items() if key.startswith("eval_")}
    for result in reconstruction_results
]
if len(reconstruction_metric_dicts) == 1:
    reconstruction_summary = {
        key: {"mean": float(value), "std": 0.0}
        for key, value in reconstruction_metric_dicts[0].items()
    }
else:
    reconstruction_summary = summarize_metrics_over_seeds(reconstruction_metric_dicts)

print("Per-seed Task 2 reconstruction metrics:")
for metrics in reconstruction_results:
    metric_text = " | ".join(
        f"{key}={value:.4f}"
        for key, value in metrics.items()
        if key.startswith("eval_")
    )
    print(
        f"seed={metrics['seed']:02d} | "
        f"matched_images={metrics['matched_images']} | "
        f"{metric_text} | fake_root={metrics['fake_root']}"
    )

print("\nSummary over available seeds:")
for key, stats in reconstruction_summary.items():
    print(f"{key}: {stats['mean']:.4f} ± {stats['std']:.4f}")

if missing_seeds:
    print("\nMissing seeds:")
    for seed, reason in missing_seeds:
        print(f"seed={seed:02d}: {reason}")